In [ ]:
import os, shutil
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from google.colab import drive

# ============================
# 1. Mount Google Drive
# ============================
drive.mount('/content/drive')

# Paths
base_dir = '/content/drive/My Drive/Stroke/main'
stroke_dir = os.path.join(base_dir, 'Stroke')
non_stroke_dir = os.path.join(base_dir, 'No Stroke')

# Output directory for split dataset
output_base = '/content/drive/My Drive/Stroke_split'
train_dir = os.path.join(output_base, 'train')
val_dir = os.path.join(output_base, 'val')
test_dir = os.path.join(output_base, 'test')

# Create folders
for folder in [train_dir, val_dir, test_dir]:
    for cls in ['Stroke', 'No Stroke']:
        os.makedirs(os.path.join(folder, cls), exist_ok=True)

# ============================
# 2. Split dataset (80/10/10)
# ============================
def split_and_copy(class_dir, class_name):
    files = os.listdir(class_dir)
    train, temp = train_test_split(files, test_size=0.2, random_state=42)  # 80% train
    val, test = train_test_split(temp, test_size=0.5, random_state=42)    # 10% val, 10% test

    for f in train:
        shutil.copy(os.path.join(class_dir, f), os.path.join(train_dir, class_name, f))
    for f in val:
        shutil.copy(os.path.join(class_dir, f), os.path.join(val_dir, class_name, f))
    for f in test:
        shutil.copy(os.path.join(class_dir, f), os.path.join(test_dir, class_name, f))

split_and_copy(stroke_dir, 'Stroke')
split_and_copy(non_stroke_dir, 'No Stroke')

print("✅ Dataset split into 80% train, 10% val, 10% test")

# ============================
# 3. Data Generators
# ============================
img_width, img_height = 150, 150
batch_size = 32

train_datagen = ImageDataGenerator(rescale=1./255, shear_range=0.2, zoom_range=0.2, horizontal_flip=True)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(img_width, img_height), batch_size=batch_size, class_mode='binary'
)
val_generator = val_test_datagen.flow_from_directory(
    val_dir, target_size=(img_width, img_height), batch_size=batch_size, class_mode='binary'
)
test_generator = val_test_datagen.flow_from_directory(
    test_dir, target_size=(img_width, img_height), batch_size=batch_size, class_mode='binary', shuffle=False
)

# ============================
# 4. Define CNN Model
# ============================
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_width, img_height, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# ============================
# 5. Train the model
# ============================
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=1
)

# ============================
# 6. Evaluate Model
# ============================
train_loss, train_acc = model.evaluate(train_generator)
val_loss, val_acc = model.evaluate(val_generator)
test_loss, test_acc = model.evaluate(test_generator)

# ============================
# 7. Save Model
# ============================
model.save('/content/drive/My Drive/Stroke/stroke_model01.h5')

# ============================
# 8. Print Summary Accuracies
# ============================
print("\n✅ Model saved to /content/drive/My Drive/Stroke/stroke_model01.h5")
print(f"Final Training Accuracy   : {train_acc*100:.2f}%")
print(f"Final Validation Accuracy : {val_acc*100:.2f}%")
print(f"Final Test Accuracy       : {test_acc*100:.2f}%")
